# E-Commerce Data Cleaning & Preparation

**Objective:** Clean, preprocess, and validate the E-Commerce Sales & Orders dataset for downstream analysis.

## Workflow

1. Load and Inspect the Dataset
2. Introduce Practice Scenarios (Intentional errors for demonstration)
3. Data Quality Assessment
4. Handling Missing Values Professionally
5. Data Cleaning and Formatting
6. Business Validation Checks
7. Change Log Documentation
8. Validation of Final Dataset
9. Export Cleaned Data


In [1]:
import pandas as pd
import numpy as np
import warnings

# Suppress minor warnings for a clean presentation
warnings.filterwarnings('ignore')

# Set options to display all columns
pd.set_option('display.max_columns', None)


## 1. Load and Inspect the Dataset

We read the raw Excel file and check its shape, types, and basic statistics.


In [2]:
# Load the dataset
# Notice: 'E-Commerce Sales & Orders.xlsx' must be in the same directory.
df_raw = pd.read_excel('E-Commerce Sales & Orders.xlsx')

print(f"Dataset Shape: {df_raw.shape[0]} rows, {df_raw.shape[1]} columns\n")
print("--- Data Types ---")
print(df_raw.dtypes)


Dataset Shape: 1200 rows, 14 columns

--- Data Types ---
OrderID                    object
Date               datetime64[ns]
CustomerID                 object
Product                    object
Quantity                    int64
UnitPrice                 float64
ShippingAddress            object
PaymentMethod              object
OrderStatus                object
TrackingNumber             object
ItemsInCart                 int64
CouponCode                 object
ReferralSource             object
TotalPrice                float64
dtype: object


In [3]:
# Inspect first 5 records to understand the structure
display(df_raw.head())


,OrderID,Date,CustomerID,Product,Quantity,UnitPrice,ShippingAddress,PaymentMethod,OrderStatus,TrackingNumber,ItemsInCart,CouponCode,ReferralSource,TotalPrice
0,ORD200000,2023-01-04,C72649,Monitor,5,570.62,928 Main St,Debit Card,Shipped,TRK37947903,7,SAVE10,Instagram,2853.10
1,ORD200001,2024-08-23,C75739,Phone,2,151.35,823 Main St,Online,Shipped,TRK91186779,3,SAVE10,Referral,302.70
2,ORD200002,2024-02-27,C81728,Tablet,5,550.68,512 Main St,Credit Card,Cancelled,TRK42903982,8,FREESHIP,Email,2753.40
3,ORD200003,2023-10-15,C33540,Chair,1,273.19,275 Main St,Debit Card,Returned,TRK62788070,5,SAVE10,Facebook,273.19
4,ORD200004,2025-05-08,C81840,Printer,4,626.01,668 Main St,Online,Delivered,TRK29241424,8,SAVE10,Email,2504.04


In [4]:
# Check memory usage and exact column info
df_raw.info(memory_usage='deep')


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1200 entries, 0 to 1199
Data columns (total 14 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   OrderID          1200 non-null   object        
 1   Date             1200 non-null   datetime64[ns]
 2   CustomerID       1200 non-null   object        
 3   Product          1200 non-null   object        
 4   Quantity         1200 non-null   int64         
 5   UnitPrice        1200 non-null   float64       
 6   ShippingAddress  1200 non-null   object        
 7   PaymentMethod    1200 non-null   object        
 8   OrderStatus      1200 non-null   object        
 9   TrackingNumber   1200 non-null   object        
 10  ItemsInCart      1200 non-null   int64         
 11  CouponCode       891 non-null    object        
 12  ReferralSource   1200 non-null   object        
 13  TotalPrice       1200 non-null   float64       
dtypes: datetime64[ns](1), float64(2), int64(

In [5]:
# Generate summary statistics for numerical columns
display(df_raw.describe())


,Date,Quantity,UnitPrice,ItemsInCart,TotalPrice
count,1200,1200.000000,1200.000000,1200.000000,1200.000000
mean,2024-03-22 16:58:48,2.945833,356.412750,5.485000,1053.968300
min,2023-01-01 00:00:00,1.000000,11.390000,1.000000,11.390000
25%,2023-08-03 18:00:00,2.000000,186.062500,4.000000,410.520000
50%,2024-03-23 00:00:00,3.000000,364.210000,5.000000,823.615000
75%,2024-11-08 12:00:00,4.000000,521.570000,7.000000,1578.475000
max,2025-06-30 00:00:00,5.000000,699.930000,10.000000,3456.400000
std,NaN,1.407557,197.177146,2.281983,819.856558


In [6]:
# Explore categorical columns
print("--- Unique values in categorical columns ---")
for col in ['Product', 'PaymentMethod', 'OrderStatus']:
    print(f"{col}: {df_raw[col].unique()}")


--- Unique values in categorical columns ---
Product: ['Monitor' 'Phone' 'Tablet' 'Chair' 'Printer' 'Laptop' 'Desk']
PaymentMethod: ['Debit Card' 'Online' 'Credit Card' 'Gift Card' 'Cash']
OrderStatus: ['Shipped' 'Cancelled' 'Returned' 'Delivered' 'Pending']


## 2. Create Additional Practice Scenarios

_Requirement: Intentionally create duplicate rows, inconsistent text formatting, and formatting problems, then clean them using Pandas._

We will inject these issues into a working copy of our dataframe (`df`) to simulate messy, real-world data problems.


In [7]:
# Create a working copy
df = df_raw.copy()

# A. Intentionally create duplicate rows
duplicates_to_inject = df.sample(5, random_state=42)
df = pd.concat([df, duplicates_to_inject], ignore_index=True)

# B. Intentionally create inconsistent text formatting
df.loc[0, 'Product'] = '   lApToP   '
df.loc[1, 'PaymentMethod'] = 'credit  card'
df.loc[2, 'OrderStatus'] = '  SHIPPED '

# C. Intentionally create formatting and logic problems
# Negative quantity
df.loc[3, 'Quantity'] = -5 
# Invalid unit price
df.loc[4, 'UnitPrice'] = -150.0 
# TotalPrice mismatch (Quantity * UnitPrice != TotalPrice)
df.loc[5, 'TotalPrice'] = 99999.99 
# Corrupt Date Format
df.loc[6, 'Date'] = '2023/15/35' # Invalid date

print(f"Injected intentional errors. New shape: {df.shape}")


Injected intentional errors. New shape: (1205, 14)


## 3. Complete Data Quality Assessment

Let's assess the quality of our newly "messy" dataframe to see exactly what we need to clean.


In [8]:
# Detect Missing Values
missing_counts = df.isnull().sum()
print("--- Missing Values ---")
print(missing_counts[missing_counts > 0])


--- Missing Values ---
CouponCode    310
dtype: int64


In [9]:
# Detect Duplicate Rows
num_duplicates = df.duplicated().sum()
print(f"\nTotal duplicate rows detected: {num_duplicates}")



Total duplicate rows detected: 5


In [10]:
# Identify inconsistent text formatting
print("\n--- Sample of Text Column Anomalies ---")
display(df[['Product', 'PaymentMethod', 'OrderStatus']].head())



--- Sample of Text Column Anomalies ---


,Product,PaymentMethod,OrderStatus
0,lApToP,Debit Card,Shipped
1,Phone,credit card,Shipped
2,Tablet,Credit Card,SHIPPED
3,Chair,Debit Card,Returned
4,Printer,Online,Delivered


In [11]:
# Detect invalid or suspicious numeric values
invalid_quantities = df[df['Quantity'] < 0]
invalid_prices = df[df['UnitPrice'] < 0]

print(f"Rows with negative Quantity: {len(invalid_quantities)}")
print(f"Rows with negative UnitPrice: {len(invalid_prices)}")


Rows with negative Quantity: 1
Rows with negative UnitPrice: 1


In [12]:
# Validate business logic (TotalPrice = Quantity * UnitPrice)
mismatch_check = df[abs((df['Quantity'] * df['UnitPrice']) - df['TotalPrice']) > 0.01]
print(f"Rows failing business logic (TotalPrice mismatch): {len(mismatch_check)}")


Rows failing business logic (TotalPrice mismatch): 3


## 4. Handle Missing Values Professionally

**Focus:** The `CouponCode` column contains missing values (NaNs).

**Decision:** We will **replace** null values with the string `'NO_COUPON'`.

**Why?** In E-commerce logic, a missing coupon code isn't an "unknown" variable that requires statistical imputation (like filling with a mean). It explicitly indicates the customer did not apply a promotional code during checkout. By preserving this explicit state as `'NO_COUPON'`, we can easily group the data later to compare the purchasing behavior of users who applied coupons versus those who didn't.


In [13]:
# Handling missing values
df['CouponCode'] = df['CouponCode'].fillna('NO_COUPON')

print(f"Missing values in CouponCode after handling: {df['CouponCode'].isnull().sum()}")


Missing values in CouponCode after handling: 0


## 5. Perform Data Cleaning

We will now fix the intentional errors, standardizing text and formatting.


In [14]:
# 1. Remove duplicate rows
initial_rows = len(df)
df = df.drop_duplicates()
print(f"Dropped {initial_rows - len(df)} duplicate rows.")


Dropped 5 duplicate rows.


In [15]:
# 2. Standardize text formatting & remove extra spaces
text_cols = ['Product', 'PaymentMethod', 'OrderStatus', 'ShippingAddress', 'ReferralSource']

for col in text_cols:
    if df[col].dtype == 'object':
        # Convert to Title Case and strip leading/trailing whitespace
        df[col] = df[col].astype(str).str.strip().str.title()
        # Replace multiple internal spaces with a single space
        df[col] = df[col].str.replace(r'\s+', ' ', regex=True)

print("Text columns standardized. Before/After preview:")
display(df[['Product', 'PaymentMethod', 'OrderStatus']].head())


Text columns standardized. Before/After preview:


,Product,PaymentMethod,OrderStatus
0,Laptop,Debit Card,Shipped
1,Phone,Credit Card,Shipped
2,Tablet,Credit Card,Shipped
3,Chair,Debit Card,Returned
4,Printer,Online,Delivered


In [16]:
# 3. Convert and validate date formats
# Using errors='coerce' to turn our intentionally corrupted date ('2023/15/35') into NaT (Not a Time)
df['Date'] = pd.to_datetime(df['Date'], errors='coerce')

invalid_dates = df['Date'].isnull().sum()
print(f"Parsed dates. Found {invalid_dates} invalid dates (converted to NaT).")

# Drop rows with corrupted dates since they cannot be accurately imputed
df = df.dropna(subset=['Date'])
print("Dropped unrecoverable date rows.")


Parsed dates. Found 1 invalid dates (converted to NaT).
Dropped unrecoverable date rows.


In [17]:
# 4. Handle invalid/negative numeric values
# Take the absolute value for accidentally negative quantities or prices (assuming data-entry error)
df['Quantity'] = df['Quantity'].abs()
df['UnitPrice'] = df['UnitPrice'].abs()

print("Negative numerical values successfully fixed.")


Negative numerical values successfully fixed.


## 6. Perform Business Validation Checks

Now that formatting is clean, we validate the underlying business logic.


In [18]:
# 1. Ensure TotalPrice = Quantity * UnitPrice
# Recalculate TotalPrice to strictly enforce business logic
df['TotalPrice_Calculated'] = df['Quantity'] * df['UnitPrice']
mismatches = df[abs(df['TotalPrice'] - df['TotalPrice_Calculated']) > 0.01]

print(f"Fixing {len(mismatches)} rows where original TotalPrice didn't match Quantity * UnitPrice.")

# Fix mismatches
df['TotalPrice'] = df['TotalPrice_Calculated']
df.drop(columns=['TotalPrice_Calculated'], inplace=True)
print("TotalPrice business logic enforced.")


Fixing 3 rows where original TotalPrice didn't match Quantity * UnitPrice.
TotalPrice business logic enforced.


In [19]:
# 2. Ensure OrderID and TrackingNumber are unique
# Duplicate OrderIDs indicate severe pipeline issues.
duplicate_orders = df[df.duplicated(subset=['OrderID'], keep=False)]

if not duplicate_orders.empty:
    print(f"Warning: Found {len(duplicate_orders)} duplicated OrderIDs. Keeping first occurrence.")
    df = df.drop_duplicates(subset=['OrderID'], keep='first')
else:
    print("Success: All OrderIDs are strictly unique.")
    
# Validate tracking numbers
duplicate_tracking = df[df.duplicated(subset=['TrackingNumber'], keep=False)]
if not duplicate_tracking.empty:
    print(f"Warning: Found {len(duplicate_tracking)} duplicate TrackingNumbers (possible grouped shipments).")
else:
    print("Success: All TrackingNumbers are unique.")


Success: All OrderIDs are strictly unique.
Success: All TrackingNumbers are unique.


## 7. Change Log Documentation

| Problem Detected      | Cleaning Action Performed               | Impact / Result                                                        |
| --------------------- | --------------------------------------- | ---------------------------------------------------------------------- |
| Missing `CouponCode`  | Filled NaNs with `'NO_COUPON'`          | Preserved logic showing no coupon was used; prevented data loss.       |
| Duplicate Rows        | Applied `.drop_duplicates()`            | Removed identical duplicates, ensuring statistical validity.           |
| Messy Text (`lApToP`) | `.str.strip().str.title()` & regex      | Clean, standardized categories ready for analysis and grouping.        |
| Invalid Dates         | `pd.to_datetime(errors='coerce')`       | Invalid dates became NaT, then dropped to ensure time-series accuracy. |
| Negative Numbers      | Applied `.abs()` to Quantities/Prices   | Fixed data entry typos, maintaining valid mathematical foundations.    |
| TotalPrice Mismatch   | Set `TotalPrice = Quantity * UnitPrice` | Enforced strict business logic and ensured financial accuracy.         |


## 8. Validate the Final Cleaned Dataset

Let's confirm the dataset is pristine before saving.


In [20]:
print("--- Final Validation Checklist ---")

print(f"1. Duplicate rows remaining: {df.duplicated().sum()}")
print(f"2. Missing values remaining:\n{df.isnull().sum()[df.isnull().sum() > 0]}")
print(f"3. Negative Quantities: {len(df[df['Quantity'] < 0])}")
print(f"4. Negative UnitPrices: {len(df[df['UnitPrice'] < 0])}")
print(f"5. TotalPrice Logic Violations: {len(df[abs((df['Quantity'] * df['UnitPrice']) - df['TotalPrice']) > 0.01])}")

print("\nDataset is fully cleaned and validated.")


--- Final Validation Checklist ---
1. Duplicate rows remaining: 0
2. Missing values remaining:
Series([], dtype: int64)
3. Negative Quantities: 0
4. Negative UnitPrices: 0
5. TotalPrice Logic Violations: 0

Dataset is fully cleaned and validated.


## 9. Save the Final Cleaned Dataset


In [21]:
output_filename = 'cleaned_ecommerce_dataset.xlsx'
df.to_excel(output_filename, index=False)
print(f"Cleaned dataset successfully saved to: {output_filename}")


Cleaned dataset successfully saved to: cleaned_ecommerce_dataset.xlsx
